# Negative-dataset filtering & inspection

Utilities used to build and sanity-check the curated negative dataset (paper, Section 5.3 / *Supervised fine-tuning*). The curated dataset is **not public** (see the paper, *Data availability*); the paths here are repo-relative placeholders - point them at your own data and weights.


### Clean Dataset

In [ ]:
# import os
# import json

# datapath = "res.cv.science.dataset.generation/datasets"
# base_json_path = "faiss/perfomance"

# total = 0
# total_trash = 0
# for dataset_name in sorted(os.listdir(datapath)):
#     dataset_num = dataset_name.split('_', 1)[1]
#     with open(f"{base_json_path}/trash_marking_{dataset_num}.json") as f:
#         marking = json.load(f)
#     num_of_trash = sum(list(marking.values()))
#     total_trash += num_of_trash
#     total += len(marking)
#     percent_trash = int(num_of_trash / len(marking) * 100)
#     print(f"Dataset {dataset_num}: {num_of_trash} / {len(marking)} ({percent_trash}%) are trash")
# print(f"\nTotal: {total_trash} / {total} ({int(total_trash / total * 100)}%) are trash")

In [ ]:
import os
import json

datapath = "res.cv.science.dataset.generation/datasets"
base_json_path = "faiss/perfomance"

good_pairs = []  # сюда сложим (dataset_num, img1, img2)

for dataset_name in sorted(os.listdir(datapath)):
    dataset_num = dataset_name.split('_', 1)[1]

    # разметка: filename -> 0 (хорошая) или 1 (мусор)
    with open(f"{base_json_path}/trash_marking_{dataset_num}.json") as f:
        marking = json.load(f)

    # ключи вида 000000001_1.jpg / 000000001_2.jpg
    by_id = {}
    for fname, label in marking.items():
        stem = fname.rsplit('.', 1)[0]          # 000000001_1
        pair_id = stem.rsplit('_', 1)[0]        # 000000001
        by_id.setdefault(pair_id, {})[fname] = label

    for pair_id, files in by_id.items():
        # ожидаем ровно две картинки в паре
        if len(files) != 2:
            continue
        fnames = sorted(files.keys())  # чтобы получить _1, _2 в предсказуемом порядке
        l1, l2 = files[fnames[0]], files[fnames[1]]

        # пара “хорошая”, если обе метки 0
        if l1 == 0 and l2 == 0:
            good_pairs.append((dataset_num, fnames[0], fnames[1]))

print(f"Всего хороших пар: {len(good_pairs)}")
# при желании можно вывести первые N
for dp, f1, f2 in good_pairs[:20]:
    print(dp, f1, f2)

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from omegaconf import OmegaConf

from src.model.model import ImageTransformPredictor
from src.dataset.tokenizer import START_TOKEN_ID, END_TOKEN_ID, PAD_TOKEN_ID

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = OmegaConf.load("configs/train_config_vit.yaml")
model_cfg = cfg.model

model = ImageTransformPredictor(model_cfg)
state = torch.load('checkpoints/itp_vit_tune_full/checkpoint_epoch_35.pth', map_location="cpu", weights_only=False)
if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]
model.load_state_dict(state, strict=False)
model.to(device).eval()

# тот же препроцессор, что в vit_encoder / датасетах
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

max_seq_len = model_cfg.decoder.max_seq_len

In [ ]:
import torch.nn.functional as F
import numpy as np

@torch.no_grad()
def loss_for_pairs(model, images1, images2, max_seq_len, device):
    """
    images1, images2: тензоры [B, 3, H, W] уже на device.
    Возвращает numpy-массив [B] со средним loss по токенам.
    """
    B = images1.size(0)

    idx = torch.full(
        (B, max_seq_len),
        PAD_TOKEN_ID,
        dtype=torch.long,
        device=device,
    )
    idx[:, 0] = START_TOKEN_ID
    if max_seq_len > 1:
        idx[:, 1] = END_TOKEN_ID

    targets = torch.roll(idx, shifts=-1, dims=1)
    targets[:, -1] = PAD_TOKEN_ID

    logits, _ = model(
        image_batch_1=images1,
        image_batch_2=images2,
        idx=idx,
        use_precomputed_embeddings=False,
    )  # [B, T, V]

    B, T, V = logits.shape
    logits_flat = logits.view(B * T, V)
    targets_flat = targets.view(B * T)

    loss_flat = F.cross_entropy(
        logits_flat,
        targets_flat,
        ignore_index=PAD_TOKEN_ID,
        reduction="none",
    )  # [B*T]

    loss_per_token = loss_flat.view(B, T)
    mask = (targets != PAD_TOKEN_ID).float()
    loss_sum = (loss_per_token * mask).sum(dim=1)
    token_count = mask.sum(dim=1).clamp_min(1.0)
    return (loss_sum / token_count).cpu().numpy()

In [ ]:
from tqdm import tqdm

all_losses = []
meta = []  # сюда сложим (dataset_num, img1, img2)

batch_size = 32
for i in tqdm(range(0, len(good_pairs), batch_size)):
    batch = good_pairs[i:i+batch_size]

    imgs1, imgs2 = [], []
    for dataset_num, f1, f2 in batch:
        ds_dir = os.path.join(datapath) #, f"dataset_{dataset_num}", "dataset")  # подстрой, если путь другой
        p1 = os.path.join(ds_dir, f1)
        p2 = os.path.join(ds_dir, f2)

        img1 = preprocess(Image.open(p1).convert("RGB"))
        img2 = preprocess(Image.open(p2).convert("RGB"))
        imgs1.append(img1)
        imgs2.append(img2)
        meta.append((dataset_num, f1, f2))

    images1 = torch.stack(imgs1).to(device)
    images2 = torch.stack(imgs2).to(device)

    batch_losses = loss_for_pairs(model, images1, images2, max_seq_len, device)
    all_losses.extend(batch_losses.tolist())

len(all_losses), len(meta)

In [ ]:
import numpy as np

losses_np = np.array(all_losses, dtype=np.float32)

percentile = 0.9  # например, верхние 5% считаем плохими (подстрой)
threshold = float(np.quantile(losses_np, percentile))
threshold

In [ ]:
good_after_loss = []
bad_by_loss = []

for (dataset_num, f1, f2), loss_value in zip(meta, losses_np):
    if loss_value <= threshold:
        good_after_loss.append((dataset_num, f1, f2, float(loss_value)))
    else:
        bad_by_loss.append((dataset_num, f1, f2, float(loss_value)))

len(good_after_loss), len(bad_by_loss)

In [ ]:
import json

with open("good_pairs_after_loss_90.json", "w", encoding="utf-8") as f:
    json.dump(
        [{"dataset": d, "img1": f1, "img2": f2, "loss": l} for d, f1, f2, l in good_after_loss],
        f, ensure_ascii=False, indent=2,
    )

with open("good_pairs_bad_by_loss_90.json", "w", encoding="utf-8") as f:
    json.dump(
        [{"dataset": d, "img1": f1, "img2": f2, "loss": l} for d, f1, f2, l in bad_by_loss],
        f, ensure_ascii=False, indent=2,
    )

In [ ]:
import json

with open("bad_pairs_list.json", "w", encoding="utf-8") as f:
    json.dump(
        [{"dataset": d, "img1": f1, "img2": f2, "loss": l} for d, f1, f2, l in bad_by_loss],
        f,
        ensure_ascii=False,
        indent=2,
    )

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt

datapath = "res.cv.science.dataset.generation/datasets"  # как раньше

def show_bad_pairs(bad_by_loss, n=8, random_seed=42):
    """
    Семплит n плохих пар и рисует их в виде картинок.
    Каждая строка: [img1 | img2], заголовок – dataset, имена файлов и loss.
    """
    if not bad_by_loss:
        print("Список bad_by_loss пуст.")
        return

    rng = random.Random(random_seed)
    samples = rng.sample(bad_by_loss, min(n, len(bad_by_loss)))

    n_rows = len(samples)
    fig, axes = plt.subplots(n_rows, 2, figsize=(8, 3 * n_rows))
    if n_rows == 1:
        axes = [axes]  # чтобы можно было итерироваться

    for ax_row, (dataset_num, f1, f2, loss_val) in zip(axes, samples):
        ds_dir = os.path.join(datapath)
        p1 = os.path.join(ds_dir, f1)
        p2 = os.path.join(ds_dir, f2)

        img1 = Image.open(p1).convert("RGB")
        img2 = Image.open(p2).convert("RGB")

        title = f"ds_{dataset_num} | {f1} / {f2}\nloss={loss_val:.4f}"

        ax_row[0].imshow(img1)
        ax_row[0].axis("off")
        ax_row[0].set_title(title, fontsize=9)

        ax_row[1].imshow(img2)
        ax_row[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_bad_pairs(bad_by_loss, n=50, random_seed=1)